# LIBERO probe (launcher)

Starts SmolVLA and runs `validation/libero_probe.py` — closed-loop policy vs LIBERO, no planner.

## 1 — Secrets, clone

In [ ]:
import getpass, os

github_token = getpass.getpass('GitHub PAT (repo scope): ')
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [ ]:
import os

if os.path.exists('/content/Longor'):
    os.chdir('/content/Longor')
    print('pulling repo...')
    !git pull
else:
    os.environ['CLONE_URL'] = f"https://{github_token}@github.com/pmish11/Longor-Robotics.git"
    !git clone $CLONE_URL "/content/Longor"
    os.chdir('/content/Longor')

## 2 — Headless rendering + app deps

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

!apt-get update
!apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r deps/requirements-libero.txt
%pip install -e .

## 3 — SmolVLA venv

In [ ]:
import os, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'

def vla_pip(*args):
    subprocess.run([VLA_PYTHON, '-m', 'pip', *args], check=True)

%pip install --quiet virtualenv
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
vla_pip('install', '-U', 'pip')
vla_pip('install', '-r', 'deps/requirements-vla-smolvla-linux.txt')
vla_pip('install', '-e', '.')

## 4 — `.env` + start VLA server + health

In [ ]:
import os, subprocess, time, urllib.request

with open('.env', 'w', encoding='utf-8') as f:
    f.write('VLA_POLICY=smolvla\nMUJOCO_GL=egl\nLOG_LEVEL=DEBUG\n')

VLA_LOG = '/tmp/libero_probe_vla_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'
log = open(VLA_LOG, 'w', buffering=1)
proc = subprocess.Popen(
    [vla_python, 'services/smolvla_service.py'],
    cwd='/content/Longor',
    stdout=log,
    stderr=subprocess.STDOUT,
    env={**os.environ, 'VLA_POLICY': 'smolvla'},
)

health_ok = False
for elapsed in range(0, 180, 5):
    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        print('Health:', resp.read().decode())
        health_ok = True
        break
    except Exception:
        print(f'Waiting for /health ({elapsed}s)...')
        time.sleep(5)

if not health_ok:
    raise RuntimeError(f'VLA server not healthy; see {VLA_LOG}')

## 5 — Run probe

In [ ]:
RUN_SCRIPT = 'validation/libero_probe.py'
# run & auto-reply to LIBERO's interactive question
!echo -e "n\n" | python {RUN_SCRIPT}